## First run this in dsmlp 
launch-sp26-cuda128.sh -l gpu-class=medium -W CSE151B_SP26_A00 -g 1 -c 8 -m 32 -v a30
##
And check you have enough space to run the model from root dir

# CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via Transformer (INT8 quantized)
4. NOTE: DO NOT RUN ON vLLM. The team has notorious trouble with vLLM. ONLY RUN WITH Transformer.
5. Scoring responses against ground-truth answers
6. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

## 1. Import modules and packages

## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [1]:
import os
os.environ["VLLM_USE_V1"] = "0"

In [2]:
#you might need to change DATA_PATH, OUTPUT_PATH
import json
import os

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0"                    
DATA_PATH   = "data/private.jsonl"
OUTPUT_PATH = "results/starter_results.jsonl"
MAX_TOKENS  = 32768                 

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

import re
import sys
from pathlib import Path
from typing import Optional
from fractions import Fraction


from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm
from collections import Counter

In [3]:
#check what gpu you have and gpu id number
!nvidia-smi

Tue May 26 20:45:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.211.01             Driver Version: 570.211.01     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L40S                    Off |   00000000:61:00.0 Off |                    0 |
| N/A   30C    P0             74W /  350W |       0MiB /  46068MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [4]:
data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

Loaded 943 questions  (300 MCQ, 643 free-form)

── MCQ sample ──
{
  "question": "Assuming the weights corresponding to the sign values are reduced by 1/10, then the arithmetic mean is ().",
  "options": [
    "Unchanged",
    "Increased by ten percent",
    "Reduced by one percent",
    "Increased by one percent",
    "Decreased by ten percent",
    "Halved",
    "Unable to determine",
    "Doubled",
    "Decreased by five percent",
    "Expanded tenfold"
  ],
  "id": 1
}

── Free-form sample ──
{
  "question": "Use the order of operations to simplify: a) $[13-(11-11)]-[8-(5-6)]=$ [ANS]\nb) $4 \\cdot 3-2+2 \\cdot 3=$ [ANS]",
  "id": 0
}


## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

### Also implementing Fewshots

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [5]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. "
    "Give EXACT answers where possible: fractions or exact expressions like (1/2)^(36/31), not decimals. "
    "If decimals are required, use as many significant figures as possible, never round to fewer. "
    "Keep ALL intermediate calculations to as many significant figures as possible to avoid rounding errors. "
    "COUNT how many values the question asks for and put ALL of them in ONE \\boxed{}. "
    "For example, if the answer is 3.14159 and 2.71828, write \\boxed{3.14159, 2.71828}. "
    "NEVER put intermediate results in \\boxed{}. "
    "Only one \\boxed{} in your entire response, at the very end."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Choose the correct option. "
    "Your final answer must be only \\boxed{X} where X is the option letter. "
    "Do not second-guess your answer. "
    "Only one \\boxed{} in your response."
)

EXAMPLES = """Example 1 (MCQ)
Q: If f(x)=2x, what is f(3)?
A. 4
B. 5
C. 6
D. 7
Answer: \\boxed{C}

Example 2 (Free-form)
Q: Compute 2 + 3.
Solution: 2 + 3 = 5.
Final answer: \\boxed{5}
"""

def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    examples_text = EXAMPLES
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        user = examples_text + "\n\n" + f"{question}\n\nOptions:\n{opts_text}"
        return SYSTEM_PROMPT_MCQ, user
    user = examples_text + "\n\n" + question
    return SYSTEM_PROMPT_MATH, user


# Verify with samples
for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} user prompt (first 200 chars) ──")
    print(usr_p[:200], "...\n")

── MCQ user prompt (first 200 chars) ──
Example 1 (MCQ)
Q: If f(x)=2x, what is f(3)?
A. 4
B. 5
C. 6
D. 7
Answer: \boxed{C}

Example 2 (Free-form)
Q: Compute 2 + 3.
Solution: 2 + 3 = 5.
Final answer: \boxed{5}


Assuming the weights correspo ...

── Free-form user prompt (first 200 chars) ──
Example 1 (MCQ)
Q: If f(x)=2x, what is f(3)?
A. 4
B. 5
C. 6
D. 7
Answer: \boxed{C}

Example 2 (Free-form)
Q: Compute 2 + 3.
Solution: 2 + 3 = 5.
Final answer: \boxed{5}


Use the order of operations t ...



## 5. Load Model with vLLM (for general case, vLLM is faster)


We load **Qwen3-4B-Thinking-2507** with **INT8 quantization** via BitsAndBytes.  
Setting `load_format="bitsandbytes"` tells vLLM to apply on-the-fly INT8 weight quantization, roughly halving GPU memory usage compared to BF16.

Key parameters:
- `gpu_memory_utilization` — fraction of GPU VRAM reserved for the model and KV cache
- `max_model_len` — maximum sequence length (prompt + generation)
- `max_num_seqs` — maximum number of sequences processed in parallel

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

llm = LLM(
    model=MODEL_ID,
    #quantization="bitsandbytes", # Using bfloat16 for better compatibility with Ampere GPUs (A30)
    #load_format="bitsandbytes",
    dtype="bfloat16",
    enable_prefix_caching=False,
    gpu_memory_utilization=0.9, # A30 has 24GB VRAM, can afford to use more
    max_model_len=16384,
    trust_remote_code=True,
    max_num_seqs=32,
    max_num_batched_tokens=131072,
)

sampling_params = SamplingParams(
    n=4,
    max_tokens=8192,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    presence_penalty=0.0,
    repetition_penalty=1.0,
    logprobs=1,
)

print("Model loaded.")

INFO 05-26 20:45:04 [utils.py:233] non-default args: {'trust_remote_code': True, 'dtype': 'bfloat16', 'max_model_len': 16384, 'enable_prefix_caching': False, 'max_num_batched_tokens': 131072, 'max_num_seqs': 32, 'disable_log_stats': True, 'model': 'Qwen/Qwen3-4B-Thinking-2507'}


WARNING 05-26 20:45:04 [envs.py:1744] Unknown vLLM environment variable detected: VLLM_USE_V1


INFO 05-26 20:45:18 [model.py:549] Resolved architecture: Qwen3ForCausalLM


INFO 05-26 20:45:18 [model.py:1678] Using max model len 16384


INFO 05-26 20:45:18 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=131072.


...

(EngineCore pid=852) 

INFO 05-26 20:46:19 [gpu_worker.py:597] CUDA graph pool memory: 0.13 GiB (actual), 0.14 GiB (estimated), difference: 0.01 GiB (8.8%).


(EngineCore pid=852) 

INFO 05-26 20:46:19 [core.py:283] init engine (profile, create kv cache, warmup model) took 44.02 seconds


Model loaded.


## 6. Generate Responses

We format every question into a chat-template prompt, then call `llm.generate()` in one batched pass.  
vLLM handles batching and scheduling internally — no manual batching needed.

In [7]:
# Build prompts for first 10 entries
prompts = []
for item in data:
    system, user = build_prompt(item["question"], item.get("options"))
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True,
    )
    prompts.append(prompt_text)

# Generate
print(f"Generating responses for {len(prompts)} questions...")
outputs = llm.generate(prompts, sampling_params=sampling_params)

def pick_best_response(output):
    """Pick response with highest cumulative log probability"""
    def score(o):
        if not o.logprobs:
            return float('-inf')
        total = 0
        for step in o.logprobs:
            # step is a dict of {token_id: Logprob}
            # the chosen token's logprob is the max (or look for the actual token)
            total += max(v.logprob for v in step.values())
        return total
    
    best = max(output.outputs, key=score)
    return best.text.strip()
    
responses = [pick_best_response(out) for out in outputs]

# Used for majority-vote (not very good ~50% accuracy)
# def extract_boxed(text):
#     # Strip thinking block
#     text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL)
#     match = re.search(r'\\boxed\{([^}]+)\}', text)
#     return match.group(1).strip() if match else None
    
# def get_best_response(out):
#     """Return the full text of the majority-voted answer."""
#     answers = [(extract_boxed(o.text), o.text) for o in out.outputs]
#     answers = [(a, t) for a, t in answers if a is not None]
#     if not answers:
#         return out.outputs[0].text  # fallback
#     best_answer = Counter(a for a, t in answers).most_common(1)[0][0]
#     # return full text of first response matching the majority answer
#     for a, t in answers:
#         if a == best_answer:
#             return t
#     return out.outputs[0].text

# responses = [get_best_response(out) for out in outputs]
    
# Preview first 3
for i in range(min(3, len(responses))):
    print(f"\n── Response {i} (id={data[i].get('id')}) ──")
    print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

Generating responses for 943 questions...


Rendering prompts:   0%|          | 0/943 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/3772 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…

## 7. Score Responses

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [ ]:
# def strip_thinking(text: str) -> str:
#     text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL)
#     text = re.sub(r'\n{3,}', '\n\n', text)
#     return re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()
    
# def latex_to_numeric(text: str) -> str:
#     """Convert \\frac{a}{b} and \\dfrac{a}{b} to decimal strings"""
#     def replace_frac(m):
#         try:
#             num, den = int(m.group(1)), int(m.group(2))
#             return str(float(Fraction(num, den)))
#         except:
#             return m.group(0)
#     text = re.sub(r'\\d?frac\{(\d+)\}\{(\d+)\}', replace_frac, text)
#     return text
    
# def extract_last_boxed(text: str) -> str:
#     """Extract last \\boxed{} content, handling nested braces like \\frac{5}{8}"""
#     results = []
#     i = 0
#     while i < len(text):
#         if text[i:i+7] == r'\boxed{':
#             depth = 0
#             start = i + 7
#             j = start
#             while j < len(text):
#                 if text[j] == '{':
#                     depth += 1
#                 elif text[j] == '}':
#                     if depth == 0:
#                         results.append(text[start:j])
#                         break
#                     depth -= 1
#                 j += 1
#         i += 1
#     return results[-1].strip() if results else ""
    
# def extract_letter(text: str) -> str:
#     text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL)
#     matches = re.findall(r'\\boxed\{([A-Za-z])\}', text)
#     if matches:
#         return matches[-1].upper()
#     matches = re.findall(r'\b([A-Z])\b', text.upper())
#     return matches[-1] if matches else ""


# def score_mcq(response: str, gold_letter: str) -> bool:
#     return extract_letter(response) == gold_letter.strip().upper()


# # Load Judger for free-form scoring
# sys.path.insert(0, ".")
# from judger import Judger
# judger = Judger(strict_extract=False)

results = []
for item, response in tqdm(zip(data, responses), total=len(data), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]
    raw_response = response
    response = strip_thinking(response)
    
    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        last_boxed = extract_last_boxed(response)
        pred_converted = latex_to_numeric(last_boxed) if last_boxed else ""
        pred = f"\\boxed{{{pred_converted}}}" if pred_converted else response
        try:
            correct = judger.auto_judge(
                pred=pred,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":       item.get("id"),
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": raw_response,
        "correct":  correct,
    })

print(f"Scoring complete. {len(results)} results.")

results = []
for item, response in zip(data, responses):
    results.append({
        "id":       item.get("id"),
        "response": response,  # raw full response for submission
    })

print(f"Collected {len(results)} results")

In [ ]:
for r in results[:10]:
    predicted = extract_last_boxed(r["response"])
    print(f"\nid={r['id']}")
    print(f"  Gold:      {r['gold']}")
    print(f"  Predicted: {predicted}")
    print(f"  Correct:   {r['correct']}")

## 8. Summary

Print accuracy broken down by question type.

In [ ]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

## 9. Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [ ]:
SAVE_EVAL = False   # Set to False when running on the private test set

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(p
                      arents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

In [ ]:
# Save submission CSV
import csv
submission_path = Path("results/submission.csv")

with open(submission_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f, quoting=csv.QUOTE_ALL)
    writer.writerow(["id", "response"])
    for r in results:
        writer.writerow([r["id"], r["response"]])

print(f"Saved submission CSV to {submission_path}")

# Sanity check
import pandas as pd
df = pd.read_csv(submission_path)
print(f"Shape: {df.shape}")
print(df.head(3))